<a href="https://colab.research.google.com/github/Muen1/multilingual-health-qa-africa/blob/main/notebooks/02_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# CELL 0 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
import os

BASE_DIR = '/content/drive/MyDrive/multilingual_health_qa'
DATA_DIR = f'{BASE_DIR}/data'
PLOT_DIR = f'{BASE_DIR}/plots'
os.makedirs(PLOT_DIR, exist_ok=True)
print("Drive mounted. Data files:", os.listdir(DATA_DIR))

Mounted at /content/drive
Drive mounted. Data files: ['Train.csv', 'Val.csv', 'Test.csv', 'SampleSubmission.csv']


In [2]:
# CELL 1 — Imports
import pandas as pd
import numpy as np
import re
import unicodedata
import matplotlib.pyplot as plt
print("Imports done.")

Imports done.


In [4]:
# CELL 2 — Load raw data
train = pd.read_csv(f'{DATA_DIR}/Train.csv')
val   = pd.read_csv(f'{DATA_DIR}/Val.csv')
test  = pd.read_csv(f'{DATA_DIR}/Test.csv')

q_col = 'input'
a_col = 'output'
lang_col = 'subset'
id_col = 'ID'

print(f"Question column : {q_col}")
print(f"Answer column   : {a_col}")
print(f"Language column : {lang_col}")
print(f"ID column       : {id_col}")

Question column : input
Answer column   : output
Language column : subset
ID column       : ID


In [5]:
# CELL 3 — Text cleaning function
def clean_text(text):
    """
    Normalize unicode characters, remove control characters,
    and strip extra whitespace.
    """
    if pd.isna(text):
        return ""
    text = str(text)
    text = unicodedata.normalize('NFC', text)                        # Unicode normalization
    text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]', '', text)  # Control characters
    text = re.sub(r'\s+', ' ', text).strip()                         # Extra whitespace
    return text

# Apply to all splits
for df in [train, val, test]:
    df[q_col] = df[q_col].apply(clean_text)
    if a_col in df.columns:
        df[a_col] = df[a_col].apply(clean_text)

print("Text cleaning complete.")
print("Sample cleaned question:", train[q_col].iloc[0][:200])
print("Sample cleaned answer:  ", train[a_col].iloc[0][:200])

Text cleaning complete.
Sample cleaned question: Ɔkwan bɛn so na mmabunbɛtumi aboa wɔn mfɛfoɔ a nsa anaa nnubɔne ama wɔayɛ wɔn ayayadeɛ? Yei bi ne sɛnea wɔbɛkyekye wɔn werɛ, sɛnea wɔbɛboa wɔn ma wɔanya mmoa firi nnwumakuo a ɛfata hɔ, ne sɛnea wɔbɛsi
Sample cleaned answer:   Mmabun betumi aboa atipɛnfo a ebia nsa anaa nnubɔne ama wɔayɛ wɔn ayayadeɛ so denam: Nkate fam mmoa a wɔde bɛma na wɔagye wɔn nkate atom a wɔremmu atɛn anaasɛ wɔmfa asodi nto wɔn so. Wɔn a wɔbɛhyɛ wɔn


In [6]:
# CELL 4 — Remove bad rows
before = len(train)
train = train[
    (train[q_col].str.strip().str.len() > 0) &
    (train[a_col].str.strip().str.len() > 0)
]
print(f"Removed {before - len(train)} empty rows. Remaining: {len(train)}")

# Remove extremely long answers (> 512 words) — hurt training
train['a_len'] = train[a_col].apply(lambda x: len(x.split()))
before2 = len(train)
train = train[train['a_len'] <= 512].drop(columns=['a_len'])
print(f"Removed {before2 - len(train)} rows with >512-word answers. Remaining: {len(train)}")

Removed 1 empty rows. Remaining: 29814
Removed 0 rows with >512-word answers. Remaining: 29814


In [7]:
# CELL 5 — Build instruction prompts
# This format tells the model the language and asks it to answer
# EXPERIMENT NOTE: Experiment 5 will change this prompt format

def build_prompt_v1(question, language):
    """Version 1: Simple format used in experiments 1-4."""
    return f"Language: {language}\nQuestion: {question}\nAnswer:"

def build_prompt_v2(question, language):
    """Version 2: More explicit format used in experiment 5."""
    return (f"You are a health assistant. Answer the following health question "
            f"in {language}.\nQuestion: {question}\nAnswer:")

# Use v1 as default (v2 will be used in Notebook 3 for Experiment 5)
PROMPT_VERSION = "v1"

def build_prompt(question, language, version="v1"):
    if version == "v2":
        return build_prompt_v2(question, language)
    return build_prompt_v1(question, language)

train['input_text']  = train.apply(
    lambda r: build_prompt(r[q_col], r[lang_col], PROMPT_VERSION), axis=1)
train['target_text'] = train[a_col].astype(str)

val['input_text']    = val.apply(
    lambda r: build_prompt(r[q_col], r[lang_col], PROMPT_VERSION), axis=1)
val['target_text']   = val[a_col].astype(str)

test['input_text']   = test.apply(
    lambda r: build_prompt(r[q_col], r[lang_col], PROMPT_VERSION), axis=1)

print(f"Prompt version: {PROMPT_VERSION}")
print("\nSample v1 prompt:\n", train['input_text'].iloc[0])
print("\nSample v2 prompt:\n", build_prompt(train[q_col].iloc[0], train[lang_col].iloc[0], "v2"))

Prompt version: v1

Sample v1 prompt:
 Language: Aka_Gha
Question: Ɔkwan bɛn so na mmabunbɛtumi aboa wɔn mfɛfoɔ a nsa anaa nnubɔne ama wɔayɛ wɔn ayayadeɛ? Yei bi ne sɛnea wɔbɛkyekye wɔn werɛ, sɛnea wɔbɛboa wɔn ma wɔanya mmoa firi nnwumakuo a ɛfata hɔ, ne sɛnea wɔbɛsiw afɔbu suban ne nsɛm a nkurɔfoɔ de gu oyarefoɔ no so no kwan.
Answer:

Sample v2 prompt:
 You are a health assistant. Answer the following health question in Aka_Gha.
Question: Ɔkwan bɛn so na mmabunbɛtumi aboa wɔn mfɛfoɔ a nsa anaa nnubɔne ama wɔayɛ wɔn ayayadeɛ? Yei bi ne sɛnea wɔbɛkyekye wɔn werɛ, sɛnea wɔbɛboa wɔn ma wɔanya mmoa firi nnwumakuo a ɛfata hɔ, ne sɛnea wɔbɛsiw afɔbu suban ne nsɛm a nkurɔfoɔ de gu oyarefoɔ no so no kwan.
Answer:


In [8]:
# CELL 6 — Verify everything looks correct before saving
print("=== VERIFICATION ===")
print(f"Train rows          : {len(train)}")
print(f"Val rows            : {len(val)}")
print(f"Test rows           : {len(test)}")
print(f"Train columns       : {train.columns.tolist()}")
print(f"Null input_text     : {train['input_text'].isna().sum()}")
print(f"Null target_text    : {train['target_text'].isna().sum()}")
print(f"Empty input_text    : {(train['input_text']=='').sum()}")
print(f"Empty target_text   : {(train['target_text']=='').sum()}")

=== VERIFICATION ===
Train rows          : 29814
Val rows            : 6686
Test rows           : 2618
Train columns       : ['ID', 'input', 'output', 'subset', 'input_text', 'target_text']
Null input_text     : 0
Null target_text    : 0
Empty input_text    : 0
Empty target_text   : 0


In [9]:
# CELL 7 — Save cleaned files to Google Drive
train.to_csv(f'{DATA_DIR}/train_clean.csv', index=False)
val.to_csv(f'{DATA_DIR}/val_clean.csv',     index=False)
test.to_csv(f'{DATA_DIR}/test_clean.csv',   index=False)

print("Saved:")
print(f"  {DATA_DIR}/train_clean.csv")
print(f"  {DATA_DIR}/val_clean.csv")
print(f"  {DATA_DIR}/test_clean.csv")
print("\nAll files in data folder:", os.listdir(DATA_DIR))

Saved:
  /content/drive/MyDrive/multilingual_health_qa/data/train_clean.csv
  /content/drive/MyDrive/multilingual_health_qa/data/val_clean.csv
  /content/drive/MyDrive/multilingual_health_qa/data/test_clean.csv

All files in data folder: ['Train.csv', 'Val.csv', 'Test.csv', 'SampleSubmission.csv', 'train_clean.csv', 'val_clean.csv', 'test_clean.csv']
